In [48]:
import sys
sys.path.append("/kaggle/input/datasets/yusufhilal/full-model/Event-Extraction-Model")

In [49]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [50]:
import json
import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path
from functools import partial
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from helpers.dataset import PageDataset, combine_pages
from helpers.metrics import pick_starts_from_probs
from models.dom_extractor import DOMAwareEventExtractor
from models.field_classifier import build_features


def load_models(checkpoint_path, classifier_path, device):
    """Load both models"""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    cfg = ckpt["cfg"]["model"]

    # Boundary Model
    model = DOMAwareEventExtractor(
        text_model_name=cfg["name"],
        tag_vocab_size=len(ckpt["tag_vocab"]),
        parent_tag_vocab_size=len(ckpt["parent_tag_vocab"]),
        num_numeric_features=len(ckpt["num_cols"]),
        num_bool_features=len(ckpt["bool_cols"]),
        d_model=cfg["d_model"],
        nhead=cfg["nhead"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
        use_tag=cfg.get("use_tag", True),
        use_parent_tag=cfg.get("use_parent_tag", True),
    ).to(device)

    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(cfg["name"])

    # classifier
    field_bundle = joblib.load(classifier_path)

    return model, tokenizer, ckpt, field_bundle

In [51]:
# loading the models -----------------------------------------------
model, tokenizer, ckpt, field_bundle = load_models(
    checkpoint_path="/kaggle/input/datasets/yusufhilal/models/dom_extractor_checkpoint_v2.pt",
    classifier_path="/kaggle/input/datasets/yusufhilal/models/field_classifier_v1.joblib",
    device=device
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [52]:
print("Model loaded successfully")
print(f"Device: {device}")
print(f"Best threshold: {ckpt['best_th']}")
print(f"Num cols: {ckpt['num_cols']}")
print(f"Bool cols: {ckpt['bool_cols']}")
print(f"Field classes: {list(field_bundle['label_encoder'].classes_)}")

Model loaded successfully
Device: cuda
Best threshold: 0.12379999999999991
Num cols: ['depth', 'sibling_index', 'children_count', 'same_tag_sibling_count', 'same_text_sibling_count', 'text_length', 'word_count', 'letter_ratio', 'digit_ratio', 'whitespace_ratio', 'attribute_count']
Bool cols: ['has_link', 'link_is_absolute', 'parent_has_link', 'is_leaf', 'contains_date', 'contains_time', 'starts_with_digit', 'ends_with_digit', 'has_class', 'has_id', 'attr_has_word_name', 'attr_has_word_date', 'attr_has_word_time', 'attr_has_word_location', 'attr_has_word_link', 'text_has_word_name', 'text_has_word_date', 'text_word_time', 'text_word_description', 'text_word_location']
Field classes: ['Date', 'Description', 'Location', 'Name', 'Price', 'Time']


In [53]:
def load_data_and_prepare(csv_path, ckpt, source_name=None):
    """
    deal with missing columns
    inputs:
        - csv file path
        - checkpoint from loaded model
        - source name can be used for single page csv files
    output: dataframe ready for models
    """
    df = pd.read_csv(csv_path)
    
    # handle source column
    if "source" not in df.columns:
        name = source_name or Path(csv_path).stem
        df["source"] = name
    
    # verify required columns
    required = ["rendering_order", "text_context", "tag", "parent_tag"] + \
               ckpt["num_cols"] + ckpt["bool_cols"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. "
            f"See README for full list of required columns."
        )
    
    # sort within each source (extra safety)
    df = df.sort_values(["source", "rendering_order"]).reset_index(drop=True)
    
    return df

In [54]:
import pandas as pd
from pathlib import Path


df = load_data_and_prepare(
    csv_path="/kaggle/input/datasets/yusufhilal/full-model/Event-Extraction-Model/data/full_data.csv",
    ckpt=ckpt,
)

print(f"Sources: {df['source'].unique()}")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Sources: ['bmiglobaled_labeled' 'gpacac.net_pattern_labeled'
 'hawaiiacac.org_pattern_labeled' 'iacac.knack.com_pattern_labeled'
 'inacac.org_pattern_labeled' 'kyacac.org_pattern_labeled'
 'members.sacac.org_column_labeled' 'nacacnet.org_pattern_labeled'
 'neacac_fall.net_pattern_labeled' 'neacac_spring.net_pattern_labeled'
 'outofstatecollegefairs.org_pattern_labeled'
 'pnacac_spring.org_pattern_labeled' 'rmacac.org_pattern_labeled'
 'sacrao.org_column_labeled' 'wacac.org_pattern_labeled']
Shape: (3100, 43)
Columns: ['rendering_order', 'tag', 'attributes', 'text_context', 'depth', 'parent_index', 'parent_tag', 'text_length', 'sibling_index', 'link', 'children_count', 'same_tag_sibling_count', 'same_text_sibling_count', 'has_link', 'link_is_absolute', 'parent_has_link', 'is_leaf', 'word_count', 'letter_ratio', 'digit_ratio', 'whitespace_ratio', 'contains_date', 'contains_time', 'starts_with_digit', 'ends_with_digit', 'attribute_count', 'has_class', 'has_id', 'attr_has_word_name', 'attr

In [55]:
@torch.no_grad()
def run_dom_extractor(page_df, model, tokenizer, ckpt, device):
    """
    function to run the model
    inputs:  
    """
    cfg = ckpt["cfg"]
    inf = cfg["inference"]

    # keep sources in same order
    sources = list(page_df.groupby("source", sort=False).groups.keys())

    # make dataset and loader for the dom model
    dataset = PageDataset(
        df=page_df,
        tokenizer=tokenizer,
        tag_vocab=ckpt["tag_vocab"],
        parent_tag_vocab=ckpt["parent_tag_vocab"],
        num_cols=ckpt["num_cols"],
        bool_cols=ckpt["bool_cols"],
        mean=ckpt["num_mean"],
        std=ckpt["num_std"],
        max_tokens=cfg["model"]["max_tokens"],
    )

    loader = DataLoader(
        dataset, batch_size=1, shuffle=False, pin_memory=False,
        collate_fn=partial(combine_pages, tokenizer=tokenizer)
    )

    results = []
    for i, batch in enumerate(loader):
        enc = {k: v.to(device) for k, v in batch["enc"].items()}
        node_mask = batch["node_mask"].to(device).bool()

        # forward pass
        bio_logits = model(
            enc=enc,
            node_offsets=batch["node_offsets"],
            node_mask=node_mask,
            tag_id=batch["tag_id"].to(device),
            parent_tag_id=batch["parent_tag_id"].to(device),
            num_feats=batch["num_feats"].to(device),
            bool_feats=batch["bool_feats"].to(device),
        )

        # get indices of real nodes
        valid = torch.where(node_mask[0])[0].cpu().numpy()

        # convert logits to probabilities
        probs_full = torch.softmax(bio_logits, dim=-1)[0].cpu().numpy()
        prob_B = probs_full[:, 1]
        prob_B_valid = prob_B[valid]
        
        # find predicted event starts using prob_B and threshold
        probs_full_valid = probs_full[valid]

        start_indices = pick_starts_from_probs(
            prob_B_valid,
            threshold=ckpt["best_th"],
            nms_k=inf["nms_k"],
            min_gap=inf["min_gap"],
        )

        for s in start_indices:
            probs_full_valid[s] = [0, 1, 0]  # force B

        if len(start_indices) == 0:
            print(f"Warning: no events found for page {i}")
            results.append([])
            continue
        
        results.append({
            "source": sources[i],
            "probs_full_valid": probs_full_valid,
            "valid": valid,
            "start_indices": start_indices
        })

    return results

In [56]:
results = run_dom_extractor(df, model, tokenizer, ckpt, device)

bio_map = {0: "O", 1: "B", 2: "I"}

for result in results:
    if not result:
        continue
    
    source = result["source"]
    probs_full_valid = result["probs_full_valid"]
    page = df[df["source"] == source].sort_values("rendering_order").reset_index(drop=True)
    
    pred_bio = probs_full_valid.argmax(axis=1)
    prob_B = probs_full_valid[:, 1]
    
    print(f"\n{'='*80}")
    print(f"Source: {source}")
    print(f"{'='*80}")
    
    for node_idx in range(len(page)):
        bio = bio_map[pred_bio[node_idx]]
        prob_b = f"{prob_B[node_idx]:.3f}"
        text = str(page.iloc[node_idx]["text_context"])
        print(f"{node_idx} | BIO={bio} | ProbB={prob_b} | {text}")


Source: bmiglobaled_labeled
0 | BIO=O | ProbB=0.003 | consent
1 | BIO=O | ProbB=0.003 | details
2 | BIO=O | ProbB=0.002 | [#iabv2settings#]
3 | BIO=O | ProbB=0.003 | about
4 | BIO=O | ProbB=0.003 | this website uses cookies
5 | BIO=O | ProbB=0.003 | we use cookies to personalise content and ads, to provide social media features and to analyse our traffic. we also share information about your use of our site with our social media, advertising and analytics partners who may combine it with other information that you’ve provided to them or that they’ve collected from your use of their services.
6 | BIO=O | ProbB=0.004 | [#gpc_banner_icon#]
7 | BIO=O | ProbB=0.003 | [#gpc_toast_text#]
8 | BIO=O | ProbB=0.004 | consent selection
9 | BIO=O | ProbB=0.003 | necessary
10 | BIO=O | ProbB=0.004 | preferences
11 | BIO=O | ProbB=0.003 | statistics
12 | BIO=O | ProbB=0.003 | marketing
13 | BIO=O | ProbB=0.004 | show details
14 | BIO=O | ProbB=0.003 | details
15 | BIO=O | ProbB=0.003 | necessary
16 

In [57]:
results = run_dom_extractor(df, model, tokenizer, ckpt, device)
sources = list(df.groupby("source", sort=False).groups.keys())

print(f"Pages processed: {len(results)}")
for i, result in enumerate(results):
    if result is None:
        print(f"Page {i}: no events found")
        continue
    
    source = result["source"]
    start_indices = result["start_indices"]
    page = df[df["source"] == source].sort_values("rendering_order").reset_index(drop=True)
    
    print(f"\nPage {i} ({source}): {len(start_indices)} events found")
    for page_idx in start_indices:
        print(f"  start node {page_idx}: {page.iloc[page_idx]['text_context']}")

Pages processed: 15

Page 0 (bmiglobaled_labeled): 18 events found
  start node 466: bali denpasar in-person fair
  start node 473: bali denpasar - 1 day high school visits
  start node 480: jakarta in-person fair
  start node 487: jakarta - 1 day high school visits
  start node 494: hanoi - 2 day high school visits
  start node 501: hanoi in-person fair
  start node 508: ho chi minh city - 1 day high school visits
  start node 515: ho chi minh city in-person fair
  start node 531: bali one delegate/ one meeting schedule
  start node 538: bali two delegates/one meeting schedule
  start node 545: bali two deleagtes/two meeting schedules
  start node 561: jakarta in-person fair
  start node 568: jakarta - 1 day high school visits
  start node 575: surabaya in-person fair
  start node 582: hanoi - 1 day high school visits
  start node 589: hanoi in-person fair
  start node 596: ho chi minh city in-person fair
  start node 603: ho chi minh city - 2 day high school visits

Page 1 (gpacac.ne

In [58]:
for result in results:
    if not result:
        continue
    
    source = result["source"]
    valid = result["valid"]
    probs_full_valid = result["probs_full_valid"]
    page = df[df["source"] == source].sort_values("rendering_order").reset_index(drop=True)
    
    print(f"Source: {source}")
    print(f"Page nodes: {len(page)}")
    print(f"Valid nodes: {len(valid)}")
    print(f"probs_full_valid shape: {probs_full_valid.shape}")
    print(f"valid: {valid}")

Source: bmiglobaled_labeled
Page nodes: 609
Valid nodes: 609
probs_full_valid shape: (609, 3)
valid: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 22

In [59]:
def run_field_classifier(page_df, dom_results, field_bundle):
    """
    Runs field classifier on all B+I nodes for each page.
    Returns list of dicts mapping page-local node index -> predicted field label.
    """
    all_node_labels = []
    
    for result in dom_results:
        if result is None:
            all_node_labels.append(None)
            continue
        
        source = result["source"]
        probs_full_valid = result["probs_full_valid"]
        
        # get page slice
        page = page_df[page_df["source"] == source].sort_values("rendering_order").reset_index(drop=True)
        
        # get indices of B+I nodes (not O)
        labels_bio = probs_full_valid.argmax(axis=1)  # 0=O, 1=B, 2=I
        bi_indices = [i for i, l in enumerate(labels_bio) if l != 0]
        
        if len(bi_indices) == 0:
            all_node_labels.append({})
            continue
        
        # run classifier on all B+I nodes at once
        bi_df = page.iloc[bi_indices]
        X = build_features(
            df=bi_df,
            tfidf=field_bundle["tfidf"],
            tag_columns=field_bundle["tag_columns"],
            parent_columns=field_bundle["parent_columns"],
            num_cols=field_bundle["num_cols"],
            bool_cols=field_bundle["bool_cols"],
            fit=False,
            use_tag=field_bundle.get("use_tag", True),
            use_parent_tag=field_bundle.get("use_parent_tag", True),
        )
        preds = field_bundle["clf"].predict(X)
        field_labels = field_bundle["label_encoder"].inverse_transform(preds)
        
        # map page-local index -> field label
        node_labels = {bi_indices[j]: field_labels[j] for j in range(len(bi_indices))}
        all_node_labels.append(node_labels)
    
    return all_node_labels

In [60]:
node_labels = run_field_classifier(df, results, field_bundle)

for i, (result, labels) in enumerate(zip(results, node_labels)):
    if result is None or labels is None:
        print(f"Page {i}: skipped")
        continue
    
    source = result["source"]
    probs_full_valid = result["probs_full_valid"]
    page = df[df["source"] == source].sort_values("rendering_order").reset_index(drop=True)
    
    bio_map = {0: "O", 1: "B", 2: "I"}
    pred_bio = probs_full_valid.argmax(axis=1)
    prob_B = probs_full_valid[:, 1]
    
    print(f"\n{'='*100}")
    print(f"Page {i} ({source})")
    print(f"{'='*100}")
    print(f"{'Idx':<6} {'RenderOrd':<12} {'BIO_true':<10} {'BIO_pred':<10} {'ProbB':<8} {'Field':<12} {'Text'}")
    print(f"{'-'*100}")
    
    for node_idx in range(len(page)):
        row = page.iloc[node_idx]
        bio_true = row.get("bio", "?")
        bio_pred = bio_map[pred_bio[node_idx]]
        prob_b = f"{prob_B[node_idx]:.3f}"
        field = labels.get(node_idx, "-")
        text = str(row["text_context"])[:40]
        render = row["rendering_order"]
        
        print(f"{node_idx:<6} {render:<12} {str(bio_true):<10} {bio_pred:<10} {prob_b:<8} {field:<12} {text}")


Page 0 (bmiglobaled_labeled)
Idx    RenderOrd    BIO_true   BIO_pred   ProbB    Field        Text
----------------------------------------------------------------------------------------------------
0      128          ?          O          0.003    -            consent
1      130          ?          O          0.003    -            details
2      132          ?          O          0.002    -            [#iabv2settings#]
3      134          ?          O          0.003    -            about
4      140          ?          O          0.003    -            this website uses cookies
5      141          ?          O          0.003    -            we use cookies to personalise content an
6      144          ?          O          0.004    -            [#gpc_banner_icon#]
7      145          ?          O          0.003    -            [#gpc_toast_text#]
8      153          ?          O          0.004    -            consent selection
9      157          ?          O          0.003    -        

In [61]:
def predict_events(page_df, dom_results, node_labels):
    """
    Takes DOM extractor results and node field labels,
    corrects predicted starts and builds event spans.
    Returns list of formatted event dicts.
    """
    all_events = []
    
    for result, labels in zip(dom_results, node_labels):
        if not result or labels is None:
            continue
        
        source = result["source"]
        probs_full_valid = result["probs_full_valid"]
        valid = result["valid"]
        start_indices = list(result["start_indices"])  # in valid-node space
        bio_labels = probs_full_valid.argmax(axis=1)   # 0=O, 1=B, 2=I
        N = len(bio_labels)
        
        # get page slice for text lookup later
        page = page_df[page_df["source"] == source].sort_values("rendering_order").reset_index(drop=True)

        # step 1: first start correction — check node above in valid-node space
        first = start_indices[0]
        if first > 0 and bio_labels[first - 1] != 0:  # node above is I or B
            start_indices[0] = first - 1
            bio_labels[first] = 2  # change old start from B to I

        # step 2: get reference field from first start
        # labels keys are in valid-node space
        reference_field = labels.get(start_indices[0])
        
        if reference_field is None or reference_field == "Other":
            print(f"Warning: could not determine reference field for {source}")
            continue

        # step 3: validate remaining starts
        validated_starts = [start_indices[0]]
        
        for start in start_indices[1:]:
            field = labels.get(start)
            
            if field == reference_field:
                validated_starts.append(start)
            elif labels.get(start - 1) == reference_field:
                validated_starts.append(start - 1)
                bio_labels[first] = 2  # change old start from B to I
            elif labels.get(start + 1) == reference_field:
                validated_starts.append(start + 1)
                bio_labels[first] = 2  # change old start from B to I
            else:
                print(f"Warning: discarding invalid start at node {start} for {source}")

        # step 4: build spans in valid-node space
        spans = []
        for i, start in enumerate(validated_starts):
            if i + 1 < len(validated_starts):
                end = validated_starts[i + 1]
            else:
                # last event — find last I node
                end = start
                for j in range(start, N):
                    if bio_labels[j] == 2:
                        end = j
                end = end + 1
            spans.append((start, end))

        # step 5: format events — map valid-node indices to page-local via valid[]
        for event_num, (start, end) in enumerate(spans, start=1):
            event = {"source": source, "event_number": event_num}
            field_counts = {}
            
            for valid_idx in range(start, end):
                label = labels.get(valid_idx)
                if label is None or label == "Other":
                    continue
                page_idx = int(valid[valid_idx])  # map to page-local index
                field_counts[label] = field_counts.get(label, 0) + 1
                key = f"{label}_{field_counts[label]}"
                event[key] = str(page.iloc[page_idx]["text_context"])
            
            all_events.append(event)
    
    return all_events

In [62]:
events = predict_events(df, results, node_labels)

for event in events:
    print(event)

{'source': 'bmiglobaled_labeled', 'event_number': 1, 'Location_1': 'bali denpasar in-person fair', 'Date_1': 'mar 01', 'Price_1': 'usd 2,800'}
{'source': 'bmiglobaled_labeled', 'event_number': 2, 'Location_1': 'bali denpasar - 1 day high school visits', 'Date_1': 'mar 02', 'Price_1': 'usd 850'}
{'source': 'bmiglobaled_labeled', 'event_number': 3, 'Location_1': 'jakarta in-person fair', 'Date_1': 'mar 07 - mar 08', 'Price_1': 'usd 4,400'}
{'source': 'bmiglobaled_labeled', 'event_number': 4, 'Location_1': 'jakarta - 1 day high school visits', 'Date_1': 'mar 09', 'Price_1': 'usd 850'}
{'source': 'bmiglobaled_labeled', 'event_number': 5, 'Location_1': 'hanoi - 2 day high school visits', 'Date_1': 'mar 11 - mar 12', 'Price_1': 'usd 1,600'}
{'source': 'bmiglobaled_labeled', 'event_number': 6, 'Location_1': 'hanoi in-person fair', 'Date_1': 'mar 14 - mar 15', 'Price_1': 'usd 4,400'}
{'source': 'bmiglobaled_labeled', 'event_number': 7, 'Location_1': 'ho chi minh city - 1 day high school visits

In [63]:
from inference import save_output
# save
save_output(events, "predicted_events.json")

In [64]:
from collections import defaultdict
page_events = defaultdict(list)

for event in events:
    page_events[event["source"]].append(event)

for source, evs in page_events.items():
    print(f"\nSource: {source} | {len(evs)} events")
    for ev in evs:
        # count fields excluding source and event_number
        num_fields = len([k for k in ev.keys() if k not in ("source", "event_number")])
        print(f"  Event {ev['event_number']}: {num_fields} fields — {list(ev.keys())}")


Source: bmiglobaled_labeled | 18 events
  Event 1: 3 fields — ['source', 'event_number', 'Location_1', 'Date_1', 'Price_1']
  Event 2: 3 fields — ['source', 'event_number', 'Location_1', 'Date_1', 'Price_1']
  Event 3: 3 fields — ['source', 'event_number', 'Location_1', 'Date_1', 'Price_1']
  Event 4: 3 fields — ['source', 'event_number', 'Location_1', 'Date_1', 'Price_1']
  Event 5: 3 fields — ['source', 'event_number', 'Location_1', 'Date_1', 'Price_1']
  Event 6: 3 fields — ['source', 'event_number', 'Location_1', 'Date_1', 'Price_1']
  Event 7: 3 fields — ['source', 'event_number', 'Location_1', 'Date_1', 'Price_1']
  Event 8: 3 fields — ['source', 'event_number', 'Location_1', 'Date_1', 'Price_1']
  Event 9: 3 fields — ['source', 'event_number', 'Location_1', 'Date_1', 'Price_1']
  Event 10: 3 fields — ['source', 'event_number', 'Location_1', 'Date_1', 'Price_1']
  Event 11: 3 fields — ['source', 'event_number', 'Location_1', 'Date_1', 'Price_1']
  Event 12: 3 fields — ['source',